<a href="https://colab.research.google.com/github/LeChatNoir578/FinTech-Fraud-Guard/blob/feature%2Fnlp-phishing/nlp_models/preprocesamiento_npl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub

In [ ]:
import re
import unicodedata
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# 1. Recursos de NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

stemmer = SnowballStemmer('spanish')
# Nota: Si los textos vienen mayoritariamente en inglés, puedes cambiar a stopwords.words('english')
stop_words = set(stopwords.words('spanish')).union(set(stopwords.words('english')))

# 2. Función de Limpieza y Preprocesamiento
def limpiar_y_preprocesar(texto):
    if not isinstance(texto, str):
        return ""

    # a. Minúsculas
    texto = texto.lower()

    # b. Normalización Unicode (remueve acentos/diacríticos)
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

    # c. Abstracción de Entidades (Tokens Genéricos)
    texto = re.sub(r'https?://\S+|www\.\S+', ' TOKEN_URL ', texto)
    texto = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' TOKEN_EMAIL ', texto)
    texto = re.sub(r'\b\d+\b', ' TOKEN_NUM ', texto)

    # d. Eliminar puntuación y caracteres especiales
    texto = re.sub(r'[^a-z\s_]', '', texto)

    # e. Tokenización
    tokens = word_tokenize(texto)

    # f. Stopwords y Stemming (Reducción a la raíz)
    tokens_limpios = [
        stemmer.stem(word) if not word.startswith('token_') else word
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    return ' '.join(tokens_limpios)

# 3. Carga y Procesamiento del Dataset de 100k
ARCHIVO_INPUT = 'email_dataset_100k.csv'
ARCHIVO_OUTPUT = 'dataset_texto_limpio_100k.csv'

print(f"Cargando dataset desde {ARCHIVO_INPUT}...")
df = pd.read_csv(ARCHIVO_INPUT)
print(f"Total de registros a procesar: {len(df):,}")

# Seleccionar la columna del cuerpo del correo ('body_plain')
print("Procesando las 100,000 filas de texto...")
df['texto_preprocesado'] = df['body_plain'].fillna('').apply(limpiar_y_preprocesar)

# 4. Guardar resultado
print(f"Guardando el dataset limpio en {ARCHIVO_OUTPUT}...")
df[['label', 'subject', 'body_plain', 'texto_preprocesado']].to_csv(ARCHIVO_OUTPUT, index=False)

print("\n¡Proceso completado exitosamente!")
print(df[['body_plain', 'texto_preprocesado']].head())

Cargando dataset desde email_dataset_100k.csv...
Total de registros a procesar: 100,000
Procesando las 100,000 filas de texto...
Guardando el dataset limpio en dataset_texto_limpio_100k.csv...

¡Proceso completado exitosamente!
                                          body_plain  \
0  Dear Vik,\n\nWe received invoice 765713. It wi...   
1  Dear user,\n\nYour parcel has been flagged. To...   
2  Hi Zack,\n\nHere are the notes from our meetin...   
3  Attention Bob,\n\nWe detected unusual activity...   
4  Team,\n\nNotes from Legal daily standup attach...   

                                  texto_preprocesado  
0  dear vik receiv invoic process accounts payabl...  
1  dear user parcel flagg restor access click lin...  
2  hi zack not meeting ple review attach document...  
3  attention bob detect unusual activity cloud st...  
4  team not legal daily standup attach ple updat ...  
